In [3]:
import rasterstats as rstats
import matplotlib.colors as colors
import rasterio as rio

# preprocess import numpy as np
from scipy.interpolate import griddata
from datetime import timedelta, datetime, date
import dateutil.parser
import pandas as pd
from scipy.spatial import distance_matrix
import xarray as xr
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed
/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [4]:
country = 'tanzania'## FILL WITH THE COUNTRY NAME
month = 1
data_path = f'../data/processed_data/{country}/' ## FILL WITH THE PATH of GLOFAS data
output_path = f'../data/downscaled_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)
# Processing data and configurations
shapefile_path = '../world_map/ne_50m_admin_0_countries.shp'

### Load country geometry


Load country geometry to use it to crop the GLOFAS data.

Country geometries are derived from Natural Earth, and in particular from this dataset
https://github.com/nvkelso/natural-earth-vector/blob/master/50m_cultural/ne_50m_admin_0_countries.shp

In [5]:
geometries = gpd.read_file(shapefile_path).loc[:,['ADMIN','geometry']].set_index('ADMIN')
if country == 'tanzania':
    geometries_country = geometries.loc['United Republic of Tanzania']
else:
    geometries_country = geometries.loc[f'{country.capitalize()}']

### Load GLOFAS

In [6]:
# Load river discharge from GLOFAS. Filtering out values with discharge lower than 0.5 m/s to reduce noise.

xdf = xr.open_mfdataset(data_path + '*processed.nc')
xdf_month = xdf.groupby('time.month').mean().sel(month=month).drop({'month'})
xdf_month = xdf_month.where(xdf_month.dis24>.5)

### Load and preprocess the geometries of the river watersheds
This is derived from the HYDROBASINS product of HYDROSHEDS, which provide shapefiles with hydrological basins at different spatial aggregations. These are provided over 12 different levels of resolution, depending on the level of the river (i.e. main river, tributaries, tributaries of the tributaries... and so on). (further info at https://www.hydrosheds.org/products/hydrobasins)
In particular, we picked the level 5 basins as they describe properly the entire basins of rivers in Tanzania, and level 12, the highest resolution for downscaling.

In [7]:
# Load river basins of levels 5 and 12 (highest resolution)
shp_basins_l5 = gpd.read_file('../hydrobasins/hybas_lake_af_lev05_v1c.shp')
shp_basins_l12 = gpd.read_file('../hydrobasins/hybas_lake_af_lev12_v1c.shp')

In [8]:
lakes_l5 = shp_basins_l5[shp_basins_l5.LAKE == 1].geometry # Saving lakes as mask for plotting

shp_basins_l5 = shp_basins_l5[shp_basins_l5.LAKE == 0] # Exclude lakes from basins
shp_basins_l12 = shp_basins_l12[shp_basins_l12.LAKE == 0]

country_l5_basins = gpd.clip(shp_basins_l5, geometries_country.geometry) # Clip basins over country. 
country_lakes_l5 = gpd.clip(lakes_l5, geometries_country.geometry)
country_l12_basins = gpd.clip(shp_basins_l12, geometries_country.geometry)

In the cell below, we add to the L12 dataset a column with the code of the L5 basin in which each L12 sub-basin is found.
This will make basin-scale plotting easier.
PFAF_ID is Pfafstetter ID, which is a convention for splitting watersheds: see the HYDROBASINS manual for more details. Here we take the first 5 digits of the PFAF_ID of the L12 basins because we want to extract basins that are within corresponding L5 basins.

In [9]:
country_l12_basins['L5basin'] = [int(str(i)[:5]) for i in country_l12_basins.PFAF_ID] 

In [10]:
## Load rivers geometry

rivers = gpd.read_file('../hydrobasins/HydroRIVERS_v10_af_shp/HydroRIVERS_v10_af.shp')
sel_basin = gpd.GeoDataFrame(country_l5_basins.iloc[[2]])
country_rivers = gpd.clip(rivers, geometries_country.geometry)
sel_basin_rivers = gpd.clip(country_rivers, sel_basin)

In [11]:
# For using rasterstats's methods for calculating various statistics over geometries, we need to know the geo transform of reference from one GLOFAS file
affine = rio.open(f'../data/processed_data/{country}/river_discharge_1992-01_2020-12_processed.nc').transform

In [12]:
# Calculating the mean discharge over L5 basins from GLOFAS data
mean_basins_l5 = rstats.zonal_stats(country_l5_basins.geometry, xdf_month.dis24.values, affine = affine, stats = 'mean')

In [13]:
# Calculating the mean discharge over L12 basins from GLOFAS data
mean_basins_l12 = rstats.zonal_stats(country_l12_basins.geometry, xdf_month.dis24.values, affine = affine, stats = 'mean')

In [14]:
# Adding the values as columns of the datasets
country_l12_basins['mean_river_discharge'] = [stat['mean'] for stat in mean_basins_l12]
country_l5_basins['mean_river_discharge'] = [stat['mean'] for stat in mean_basins_l5]

In [15]:
country_l5_basins.to_file(output_path+f'river_discharge_l5_{country}_m{month}.geojson', driver='GeoJSON')
country_l12_basins.to_csv(output_path+f'river_discharge_l12_{country}_m{month}.csv')